In [2]:
import pandas as pd
import os
import kagglehub
from dotenv import load_dotenv

c:\Users\kimmi\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#Load .env file for Kaggle Credentials 
load_dotenv()
username = os.getenv("KAGGLE_USERNAME")
key = os.getenv("KAGGLE_KEY")

os.environ["KAGGLE_USERNAME"] = username
os.environ["KAGGLE_KEY"] = key

In [4]:
#Locate path to the Kaggle dataset
path = kagglehub.dataset_download(
    "aiaiaidavid/the-big-dataset-of-ultra-marathon-running"
)

In [5]:
df = pd.read_csv(f"{path}/TWO_CENTURIES_OF_UM_RACES.csv", encoding="latin1")
df = df.drop(columns=["Event dates", "Event number of finishers", "Athlete club", "Athlete average speed"])

C:\Users\kimmi\AppData\Local\Temp\ipykernel_33216\1301004287.py:1: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{path}/TWO_CENTURIES_OF_UM_RACES.csv", encoding="latin1")


In [6]:
#1) Extract data since 2000s (TRY DIFFERENT THRESHOLDS)
df = df[df['Year of event'] >= 2000]

In [7]:
#2) Extract athletes that have data on a large portion of their career
# Get Athlete IDs where count exceeds 15 races (TRY DIFFERENT THRESHOLDS)
valid_athletes = df['Athlete ID'].value_counts()
valid_athletes = valid_athletes[valid_athletes > 15].index
df = df[df['Athlete ID'].isin(valid_athletes)]
#104807 rows × 8 columns

In [8]:
#Count the number of entries for each race
df['Event name'].value_counts()

Event name
Two Oceans Marathon (RSA)                                    66695
Comrades Marathon - Down Run (RSA)                           49422
Comrades Marathon - Up Run (RSA)                             45381
Two Oceans Marathon - 50km Split (RSA)                       38182
Ultra Trail Tour du Mont Blanc (UTMB) (FRA)                  15324
                                                             ...  
Universiti Malaya 6 hours Ultra Marathon (MAS)                   1
Ranscombe Autumn Challenge 8 hours run - Sunday Run (GBR)        1
Buckley's Chance 50K Trail Run (AUS)                             1
Javadhu Hills Ultra 50 Km (IND)                                  1
100 Miles Thailand Back2Back (THA)                               1
Name: count, Length: 23638, dtype: int64

In [14]:
# Two Oceans Marathon (RSA) : 121129
df_two_oceans = df[df['Event name'] == 'Two Oceans Marathon (RSA)']
df_two_oceans.sample(10)

,Year of event,Event name,Event distance/length,Athlete performance,Athlete country,Athlete year of birth,Athlete gender,Athlete age category,Athlete ID
176133,2018,Two Oceans Marathon (RSA),56km,6:07:20 h,RSA,1979.0,M,M35,136882
807146,2016,Two Oceans Marathon (RSA),56km,6:38:07 h,RSA,1965.0,M,M50,428795
178987,2018,Two Oceans Marathon (RSA),56km,6:45:08 h,RSA,1963.0,M,M50,139171
5981306,2014,Two Oceans Marathon (RSA),56km,6:28:03 h,RSA,1979.0,M,M23,106906
3873592,2000,Two Oceans Marathon (RSA),56km,5:32:04 h,RSA,1967.0,M,M23,134903
4092749,2003,Two Oceans Marathon (RSA),56km,4:46:03 h,RSA,1974.0,M,M23,14331
4941199,2010,Two Oceans Marathon (RSA),56km,5:28:02 h,RSA,1949.0,M,M60,1075458
4285951,2005,Two Oceans Marathon (RSA),56km,5:29:58 h,RSA,1946.0,M,M55,301611
180215,2018,Two Oceans Marathon (RSA),56km,6:56:03 h,RSA,1983.0,F,W35,140171
4400333,2006,Two Oceans Marathon (RSA),56km,6:44:08 h,RSA,1962.0,M,M40,415680


In [16]:
#Calculate the athletes average speed in km/hr --> Used to Extract top threshold of fastest runners
#Since format H:MM:SS h harder to compare, numerically

df_two_oceans["distance_km"] = df_two_oceans["Event distance/length"].str.extract(r"(\d+\.?\d*)").astype(float)
df_two_oceans["distance_km"]
time_parts = df_two_oceans["Athlete performance"].str.replace(" h", "").str.split(":", expand=True)

df_two_oceans["hours"] = (
    time_parts[0].astype(float) +
    time_parts[1].astype(float) / 60 +
    time_parts[2].astype(float) / 3600
)

df_two_oceans["calculated_speed"] = df_two_oceans["distance_km"] / df_two_oceans["hours"]

df_two_oceans["calculated_speed"].head()

C:\Users\kimmi\AppData\Local\Temp\ipykernel_33216\2365875622.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_two_oceans["distance_km"] = df_two_oceans["Event distance/length"].str.extract(r"(\d+\.?\d*)").astype(float)
C:\Users\kimmi\AppData\Local\Temp\ipykernel_33216\2365875622.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_two_oceans["hours"] = (
C:\Users\kimmi\AppData\Local\Temp\ipykernel_33216\2365875622.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from 

171456    17.571690
171458    17.486339
171462    17.241084
171463    17.058724
171465    16.959704
Name: calculated_speed, dtype: float64